In [16]:
import os
import io

import numpy as np

from typing import Tuple

import time
import cv2

from PIL import Image, ImageOps

import torch
import torch.nn as nn
import torchvision
import torch.onnx
import torchsummary
print('torch.__version__', torch.__version__)

import onnx
#import onnx2keras
import onnxruntime
from onnxsim import simplify
#from onnx_tf.backend import prepare
print('onnx.__version__', onnx.__version__)

# import tvm
# import tvm.relay
# import tvm.contrib.graph_runtime as graph_runtime

from mobilenet_v2_tsm import MobileNetV2
from mobilenet_v2_keras_tsm import MobileNetV2TSM

import tensorflow as tf
print('tf.__version__', tf.__version__)
from tensorflow.python.keras import layers
from tensorflow.python.keras.engine import training
from keras.models import load_model
print('tf.keras.__version__', tf.keras.__version__)

#import tensorflowjs as tfjs
#print('tfjs.__version__', tfjs.__version__)

# import warnings
# warnings.filterwarnings('ignore')

torch.__version__ 1.2.0
onnx.__version__ 1.7.0
tf.__version__ 2.2.0
tf.keras.__version__ 2.3.0-tf


In [17]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
SOFTMAX_THRES = 0
HISTORY_LOGIT = True
REFINE_OUTPUT = True

# def torch2tvm_module(torch_module: torch.nn.Module, torch_inputs: Tuple[torch.Tensor, ...], target):
#     torch_module.eval()
#     input_names = []
#     input_shapes = {}
#     with torch.no_grad():
#         for index, torch_input in enumerate(torch_inputs):
#             name = "i" + str(index)
#             input_names.append(name)
#             input_shapes[name] = torch_input.shape
#         buffer = io.BytesIO()
#         torch.onnx.export(torch_module, torch_inputs, buffer, input_names=input_names, output_names=["o" + str(i) for i in range(len(torch_inputs))])
#         outs = torch_module(*torch_inputs)
#         buffer.seek(0, 0)
#         onnx_model = onnx.load_model(buffer)
#         relay_module, params = tvm.relay.frontend.from_onnx(onnx_model, shape=input_shapes)
#     with tvm.relay.build_config(opt_level=3):
#         graph, tvm_module, params = tvm.relay.build(relay_module, target, params=params)
#     return graph, tvm_module, params


# def torch2executor(torch_module: torch.nn.Module, torch_inputs: Tuple[torch.Tensor, ...], target):
#     prefix = f"mobilenet_tsm_tvm_{target}"
#     lib_fname = f'{prefix}.tar'
#     graph_fname = f'{prefix}.json'
#     params_fname = f'{prefix}.params'
#     if os.path.exists(lib_fname) and os.path.exists(graph_fname) and os.path.exists(params_fname):
#         with open(graph_fname, 'rt') as f:
#             graph = f.read()
#         tvm_module = tvm.module.load(lib_fname)
#         params = tvm.relay.load_param_dict(bytearray(open(params_fname, 'rb').read()))
#     else:
#         graph, tvm_module, params = torch2tvm_module(torch_module, torch_inputs, target)
#         tvm_module.export_library(lib_fname)
#         with open(graph_fname, 'wt') as f:
#             f.write(graph)
#         with open(params_fname, 'wb') as f:
#             f.write(tvm.relay.save_param_dict(params))

#     ctx = tvm.gpu() if target.startswith('cuda') else tvm.cpu()
#     graph_module = graph_runtime.create(graph, tvm_module, ctx)
#     for pname, pvalue in params.items():
#         graph_module.set_input(pname, pvalue)

#     def executor(inputs: Tuple[tvm.nd.NDArray]):
#         for index, value in enumerate(inputs):
#             graph_module.set_input(index, value)
#         graph_module.run()
#         return tuple(graph_module.get_output(index) for index in range(len(inputs)))

#     return executor, ctx


# def get_executor(use_gpu=True):
#     torch_module = MobileNetV2(n_class=27)
#     if not os.path.exists("mobilenetv2_jester_online.pth.tar"):  # checkpoint not downloaded
#         print('Downloading PyTorch checkpoint...')
#         import urllib.request
#         url = 'https://file.lzhu.me/projects/tsm/models/mobilenetv2_jester_online.pth.tar'
#         urllib.request.urlretrieve(url, './mobilenetv2_jester_online.pth.tar')
#     torch_module.load_state_dict(torch.load("mobilenetv2_jester_online.pth.tar"))
#     torch_inputs = (torch.rand(1, 3, 224, 224),
#                     torch.zeros([1, 3, 56, 56]),
#                     torch.zeros([1, 4, 28, 28]),
#                     torch.zeros([1, 4, 28, 28]),
#                     torch.zeros([1, 8, 14, 14]),
#                     torch.zeros([1, 8, 14, 14]),
#                     torch.zeros([1, 8, 14, 14]),
#                     torch.zeros([1, 12, 14, 14]),
#                     torch.zeros([1, 12, 14, 14]),
#                     torch.zeros([1, 20, 7, 7]),
#                     torch.zeros([1, 20, 7, 7]))
#     if use_gpu:
#         target = 'cuda'
#     else:
#         target = 'llvm -mcpu=cortex-a72 -target=armv7l-linux-gnueabihf'
#     return torch2executor(torch_module, torch_inputs, target)


def transform(frame: np.ndarray):
    # 480, 640, 3, 0 ~ 255
    frame = cv2.resize(frame, (224, 224))  # (224, 224, 3) 0 ~ 255
    frame = frame / 255.0  # (224, 224, 3) 0 ~ 1.0
    frame = np.transpose(frame, axes=[2, 0, 1])  # (3, 224, 224) 0 ~ 1.0
    frame = np.expand_dims(frame, axis=0)  # (1, 3, 480, 640) 0 ~ 1.0
    return frame


class GroupScale(object):
    """ Rescales the input PIL.Image to the given 'size'.
    'size' will be the size of the smaller edge.
    For example, if height > width, then image will be
    rescaled to (size * height / width, size)
    size: size of the smaller edge
    interpolation: Default: PIL.Image.BILINEAR
    """

    def __init__(self, size, interpolation=Image.BILINEAR):
        self.worker = torchvision.transforms.Scale(size, interpolation)

    def __call__(self, img_group):
        return [self.worker(img) for img in img_group]


class GroupCenterCrop(object):
    def __init__(self, size):
        self.worker = torchvision.transforms.CenterCrop(size)

    def __call__(self, img_group):
        return [self.worker(img) for img in img_group]


class Stack(object):

    def __init__(self, roll=False):
        self.roll = roll

    def __call__(self, img_group):
        if img_group[0].mode == 'L':
            return np.concatenate([np.expand_dims(x, 2) for x in img_group], axis=2)
        elif img_group[0].mode == 'RGB':
            if self.roll:
                return np.concatenate([np.array(x)[:, :, ::-1] for x in img_group], axis=2)
            else:
                return np.concatenate(img_group, axis=2)


class ToTorchFormatTensor(object):
    """ Converts a PIL.Image (RGB) or numpy.ndarray (H x W x C) in the range [0, 255]
    to a torch.FloatTensor of shape (C x H x W) in the range [0.0, 1.0] """

    def __init__(self, div=True):
        self.div = div

    def __call__(self, pic):
        if isinstance(pic, np.ndarray):
            # handle numpy array
            img = torch.from_numpy(pic).permute(2, 0, 1).contiguous()
        else:
            # handle PIL Image
            img = torch.ByteTensor(torch.ByteStorage.from_buffer(pic.tobytes()))
            img = img.view(pic.size[1], pic.size[0], len(pic.mode))
            # put it from HWC to CHW format
            # yikes, this transpose takes 80% of the loading time/CPU
            img = img.transpose(0, 1).transpose(0, 2).contiguous()
        return img.float().div(255) if self.div else img.float()


class GroupNormalize(object):
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, tensor):
        rep_mean = self.mean * (tensor.size()[0] // len(self.mean))
        rep_std = self.std * (tensor.size()[0] // len(self.std))

        # TODO: make efficient
        for t, m, s in zip(tensor, rep_mean, rep_std):
            t.sub_(m).div_(s)

        return tensor


def get_transform():
    cropping = torchvision.transforms.Compose([
        GroupScale(256),
        GroupCenterCrop(224),
    ])
    transform = torchvision.transforms.Compose([
        cropping,
        Stack(roll=False),
        ToTorchFormatTensor(div=True),
        GroupNormalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    return transform

categories = [
    "Doing other things",  # 0
    "Drumming Fingers",  # 1
    "No gesture",  # 2
    "Pulling Hand In",  # 3
    "Pulling Two Fingers In",  # 4
    "Pushing Hand Away",  # 5
    "Pushing Two Fingers Away",  # 6
    "Rolling Hand Backward",  # 7
    "Rolling Hand Forward",  # 8
    "Shaking Hand",  # 9
    "Sliding Two Fingers Down",  # 10
    "Sliding Two Fingers Left",  # 11
    "Sliding Two Fingers Right",  # 12
    "Sliding Two Fingers Up",  # 13
    "Stop Sign",  # 14
    "Swiping Down",  # 15
    "Swiping Left",  # 16
    "Swiping Right",  # 17
    "Swiping Up",  # 18
    "Thumb Down",  # 19
    "Thumb Up",  # 20
    "Turning Hand Clockwise",  # 21
    "Turning Hand Counterclockwise",  # 22
    "Zooming In With Full Hand",  # 23
    "Zooming In With Two Fingers",  # 24
    "Zooming Out With Full Hand",  # 25
    "Zooming Out With Two Fingers"  # 26
]

n_still_frame = 0

def process_output(idx_, history):
    # idx_: the output of current frame
    # history: a list containing the history of predictions
    if not REFINE_OUTPUT:
        return idx_, history

    max_hist_len = 20  # max history buffer

    # mask out illegal action
    if idx_ in [7, 8, 21, 22, 3]:
        idx_ = history[-1]

    # use only single no action class
    if idx_ == 0:
        idx_ = 2
    
    # history smoothing
    if idx_ != history[-1]:
        if not (history[-1] == history[-2]): #  and history[-2] == history[-3]):
            idx_ = history[-1]
    

    history.append(idx_)
    history = history[-max_hist_len:]

    return history[-1], history

In [19]:
os.makedirs('./models', exist_ok=True)
TORCH_MODEL_PATH= './models/mobilenetv2_jester_online.pth.tar'
torch_module = MobileNetV2(n_class=27)
if not os.path.exists(TORCH_MODEL_PATH):  # checkpoint not downloaded
    print('Downloading PyTorch checkpoint...')
    import urllib.request
    url = 'https://file.lzhu.me/projects/tsm/models/mobilenetv2_jester_online.pth.tar'
    urllib.request.urlretrieve(url, TORCH_MODEL_PATH)
torch_module.load_state_dict(torch.load(TORCH_MODEL_PATH))
torch_module.eval()

MobileNetV2(
  (features): ModuleList(
    (0): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
        (3): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (4): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
       

In [309]:
torch.save(torch_module.state_dict(), './torch_module.pth')

In [20]:
from tensorflow.python.keras import backend
backend.set_image_data_format('channels_last')

In [21]:
keras_buffer = [
    np.zeros([1, 224, 224, 3]),
    np.zeros([1, 56, 56, 3]),
    np.zeros([1, 28, 28, 4]),
    np.zeros([1, 28, 28, 4]),
    np.zeros([1, 14, 14, 8]),
    np.zeros([1, 14, 14, 8]),
    np.zeros([1, 14, 14, 8]),
    np.zeros([1, 14, 14, 12]),
    np.zeros([1, 14, 14, 12]),
    np.zeros([1, 7, 7, 20]),
    np.zeros([1, 7, 7, 20])
]

keras_tsm = MobileNetV2TSM(
    input_shapes=[tensor.shape[1:] for tensor in keras_buffer],
    alpha=1.0,
    weights=None,
    input_tensor=None,
    classes=27
)

test_input = {f'i{i}': keras_buffer[i] for i in range(len(keras_buffer))}

y_pred = keras_tsm(test_input)
print(len(y_pred))
print([y_pred[i].shape for i in range(len(y_pred))])
print(y_pred[0])

[(224, 224, 3), (56, 56, 3), (28, 28, 4), (28, 28, 4), (14, 14, 8), (14, 14, 8), (14, 14, 8), (14, 14, 12), (14, 14, 12), (7, 7, 20), (7, 7, 20)]
11
[TensorShape([1, 27]), TensorShape([1, 56, 56, 4]), TensorShape([1, 28, 28, 5]), TensorShape([1, 28, 28, 5]), TensorShape([1, 14, 14, 9]), TensorShape([1, 14, 14, 9]), TensorShape([1, 14, 14, 9]), TensorShape([1, 14, 14, 13]), TensorShape([1, 14, 14, 13]), TensorShape([1, 7, 7, 21]), TensorShape([1, 7, 7, 21])]
tf.Tensor(
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0.]], shape=(1, 27), dtype=float32)


In [312]:
print(keras_tsm.output_names)
# keras_tsm.compile()
keras_tsm.summary()
keras_tsm.save(save_format= 'h5',filepath='./keras_tsm.h5')

['classifier', 'tf_op_layer_concat_10', 'tf_op_layer_concat_11', 'tf_op_layer_concat_12', 'tf_op_layer_concat_13', 'tf_op_layer_concat_14', 'tf_op_layer_concat_15', 'tf_op_layer_concat_16', 'tf_op_layer_concat_17', 'tf_op_layer_concat_18', 'tf_op_layer_concat_19']
Model: "mobilenetv2_1.00_224"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
i0 (InputLayer)                 [(None, 224, 224, 3) 0                                            
__________________________________________________________________________________________________
features.0.0 (Conv2D)           (None, 112, 112, 32) 864         i0[0][0]                         
__________________________________________________________________________________________________
features.0.1 (BatchNormalizatio (None, 112, 112, 32) 128         features.0.0[0][0]               
____________

In [23]:
torchsummary.summary(torch_module, [tensor.shape[1:][::-1] for tensor in keras_buffer])

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 112, 112]             864
       BatchNorm2d-2         [-1, 32, 112, 112]              64
             ReLU6-3         [-1, 32, 112, 112]               0
            Conv2d-4         [-1, 32, 112, 112]             288
       BatchNorm2d-5         [-1, 32, 112, 112]              64
             ReLU6-6         [-1, 32, 112, 112]               0
            Conv2d-7         [-1, 16, 112, 112]             512
       BatchNorm2d-8         [-1, 16, 112, 112]              32
  InvertedResidual-9         [-1, 16, 112, 112]               0
           Conv2d-10         [-1, 96, 112, 112]           1,536
      BatchNorm2d-11         [-1, 96, 112, 112]             192
            ReLU6-12         [-1, 96, 112, 112]               0
           Conv2d-13           [-1, 96, 56, 56]             864
      BatchNorm2d-14           [-1, 96,

In [202]:
from collections import defaultdict
from tensorflow.keras.layers import DepthwiseConv2D, Conv2D, BatchNormalization, Dense, ReLU

def tsm_pth2keras(torch_model, keras_model):
    torch_model.eval()
    
    m = {}  # {'classifier.1.bias': np.array(...), ...}

    for k, v in torch_model.named_parameters():
        m[k] = v
    for k, v in torch_model.named_buffers():  # for BatchNormalization
        if 'num_batches_tracked' in k: continue  # no analogue in keras
        m[k] = v
    
#     structured_m = defaultdict(dict)
#     for k in m:
#         splitted_name = k.split('.')
#         if len(splitted_name) > 3:  # Conv2D, BatchNorm2D
#             layer_num = splitted_name[1]
#             structured_m[f'features.{layer_num}']['.'.join(splitted_name[2:])] = m[k]
#         if len(splitted_name) == 2:  # Dense (classifier)
#             structured_m[f'classifier'][splitted_name[1]] = m[k]
            
    # for each layer in keras model search for its clone in pytorch model
    # Structure of MobileNetV2TSM:
    # > first conv layer
    # > (inv_res_block + inv_res_block_shift) x M
    # > last conv layer
    # > classifier (Dense) layer
    keras_layers_list = []
    with torch.no_grad():
        for layer in keras_model.layers:
            if isinstance(layer, DepthwiseConv2D):
                print(layer.name)
                keras_layers_list.append(layer)
#                 print(layer.weights[0].shape)
#                 print(m[layer.name+'.weight'].shape)
                weights = []
                weights.append(m[layer.name+'.weight'].permute(2, 3, 0, 1).detach().numpy()) # weight
                if layer.use_bias:
                    weights.append(m[layer.name+'.bias'].detach().numpy()) # bias
                layer.set_weights(weights)
            elif isinstance(layer, Conv2D):
                keras_layers_list.append(layer)
                # https://github.com/keras-team/keras/issues/8144
                # pth: (out_ch, in_ch, h, w)
                # tf/keras: (h, w, in_ch, out_ch)
                print(layer.name)
                weights = []
#                 print(m[layer.name+'.weight'].shape)
                weights.append(m[layer.name+'.weight'].permute(2, 3, 1, 0).detach().numpy()) # weight
                if layer.use_bias:
                    weights.append(m[layer.name+'.bias'].detach().numpy()) # bias
                layer.set_weights(weights)
            elif isinstance(layer, BatchNormalization):
                keras_layers_list.append(layer)
                print(layer.name)
                weights = []
                if layer.scale:
                    weights.append(m[layer.name+'.weight'].detach().numpy()) # gamma
                if layer.center:
                    weights.append(m[layer.name+'.bias'].detach().numpy()) # beta
                weights.append(m[layer.name+'.running_mean'].detach().numpy()) # running_mean
                weights.append(m[layer.name+'.running_var'].detach().numpy()) # running_var
                layer.set_weights(weights)
            elif isinstance(layer, Dense):
                keras_layers_list.append(layer)
                print(layer.name)
                weights = []
                weights.append(m[layer.name+'.weight'].t().detach().numpy())
                if layer.use_bias:
                    weights.append(m[layer.name+'.bias'].detach().numpy())
                layer.set_weights(weights)
    return(keras_layers_list)

In [25]:
for layer in keras_tsm.layers:
    print(layer.weights)

[]
[<tf.Variable 'features.0.0_1/kernel:0' shape=(3, 3, 3, 32) dtype=float32, numpy=
array([[[[ 0.09311377,  0.05343632, -0.08222498, -0.04430173,
          -0.10429185,  0.10618506,  0.0077446 , -0.05021825,
           0.07225071, -0.00246526,  0.09570828, -0.04255681,
          -0.02537839,  0.08788443, -0.097038  ,  0.11625777,
           0.03659712,  0.12399216,  0.00364882,  0.03097177,
          -0.00185215,  0.04594263,  0.08384965, -0.02914038,
          -0.10485101,  0.05396833, -0.00335459,  0.10483985,
           0.11760806,  0.02805774,  0.00943302, -0.02708309],
         [ 0.07159953,  0.01310506,  0.07538991,  0.13396709,
          -0.04248202, -0.04248123, -0.01356432, -0.01276404,
           0.07899112, -0.05256826, -0.04151525, -0.08152759,
           0.03311904,  0.05986999,  0.05288862, -0.05288029,
           0.11149661,  0.10862605, -0.05495218,  0.10565108,
           0.11881106, -0.06622734, -0.11000246,  0.07753238,
           0.03931133,  0.0659003 , -0.0946501

       1., 1., 1., 1., 1., 1., 1., 1.], dtype=float32)>]
[]
[<tf.Variable 'features.3.conv.3_1/depthwise_kernel:0' shape=(3, 3, 144, 1) dtype=float32, numpy=
array([[[[-0.02126046],
         [ 0.03267185],
         [-0.04410129],
         ...,
         [-0.00561459],
         [ 0.00881317],
         [ 0.02477223]],

        [[-0.06269108],
         [ 0.03812479],
         [-0.05220675],
         ...,
         [-0.05908224],
         [ 0.0405464 ],
         [-0.0073403 ]],

        [[ 0.00924311],
         [ 0.01104333],
         [ 0.02673782],
         ...,
         [ 0.04189923],
         [-0.01367253],
         [ 0.03549009]]],


       [[[-0.0438561 ],
         [-0.06406506],
         [ 0.02906698],
         ...,
         [ 0.02506189],
         [ 0.03085656],
         [-0.03637856]],

        [[ 0.01264083],
         [ 0.05999936],
         [ 0.003361  ],
         ...,
         [ 0.02698959],
         [ 0.06239498],
         [ 0.04701985]],

        [[ 0.04326226],
         [ 0.022

       1., 1., 1., 1., 1., 1., 1., 1., 1., 1.], dtype=float32)>]
[]
[<tf.Variable 'features.10.conv.6_1/kernel:0' shape=(1, 1, 384, 64) dtype=float32, numpy=
array([[[[ 7.97824860e-02, -2.28834674e-02, -3.89230400e-02, ...,
          -4.22607958e-02, -6.34406209e-02,  9.89511162e-02],
         [ 4.57085073e-02, -2.24243477e-02,  1.04588509e-01, ...,
          -6.90779462e-02, -6.49413615e-02, -1.10385172e-01],
         [-8.61075521e-02, -1.36503577e-02,  2.68205702e-02, ...,
          -6.28009439e-05, -9.96726751e-03, -1.14045613e-01],
         ...,
         [ 8.28042626e-03, -1.13664769e-01, -2.01353729e-02, ...,
           7.09518045e-03,  3.79410237e-02, -4.83440608e-03],
         [ 8.65061730e-02,  1.44853890e-02,  5.95428497e-02, ...,
          -7.20769837e-02, -1.08062044e-01, -1.11203209e-01],
         [ 7.17507750e-02, -4.02418077e-02,  4.49037403e-02, ...,
          -1.96869820e-02, -1.59358233e-02,  1.00300282e-01]]]],
      dtype=float32)>]
[<tf.Variable 'features.10.conv.7_

[<tf.Variable 'features.12.conv.1_1/gamma:0' shape=(576,) dtype=float32, numpy=
array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1

[<tf.Variable 'features.15.conv.4_1/gamma:0' shape=(960,) dtype=float32, numpy=
array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1

[<tf.Variable 'features.16.conv.1_1/gamma:0' shape=(960,) dtype=float32, numpy=
array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1

In [28]:
np.random.seed(42)

In [29]:
buffer = [
    np.random.rand(*[1, 224, 224, 3]),
    np.random.rand(*[1, 56, 56, 3]),
    np.random.rand(*[1, 28, 28, 4]),
    np.random.rand(*[1, 28, 28, 4]),
    np.random.rand(*[1, 14, 14, 8]),
    np.random.rand(*[1, 14, 14, 8]),
    np.random.rand(*[1, 14, 14, 8]),
    np.random.rand(*[1, 14, 14, 12]),
    np.random.rand(*[1, 14, 14, 12]),
    np.random.rand(*[1, 7, 7, 20]),
    np.random.rand(*[1, 7, 7, 20])
]

pth_inputs = [torch.from_numpy(x).permute(0, 3, 1, 2).float() for x in buffer]
keras_inputs = {f'i{i}': buffer[i] for i in range(len(buffer))}

pth_outputs = torch_module(*pth_inputs)
pth_outputs = [
    x.permute(0, 2, 3, 1).detach().numpy() 
    if len(x.shape) > 2 else x.detach().numpy() 
    for x in pth_outputs
]
pth_gesture, pth_buf = pth_outputs[0], pth_outputs[1:]

In [30]:
ker_outputs = keras_tsm.predict(keras_inputs)
ker_gesture, ker_buf = ker_outputs[0], ker_outputs[1:]

In [31]:
print(np.abs(pth_gesture-ker_gesture).max())

11.178182


In [32]:
for i in range(len(pth_buf)):
    print(pth_buf[i].shape)
    print(ker_buf[i][:,:,:,:-1].shape)
    print(
        (pth_buf[i]-ker_buf[i][:,:,:,:-1]).min(), 
        (pth_buf[i]-ker_buf[i][:,:,:,:-1]).max()
    )
#     print([np.abs(pth-ker[:,:,:,:-1]).max() for pth, ker in zip(pth_buf, ker_buf)])

(1, 56, 56, 3)
(1, 56, 56, 3)
-5.208493 5.3847036
(1, 28, 28, 4)
(1, 28, 28, 4)
-3.642719 2.9163022
(1, 28, 28, 4)
(1, 28, 28, 4)
-3.9536288 3.393629
(1, 14, 14, 8)
(1, 14, 14, 8)
-3.8981268 2.7521858
(1, 14, 14, 8)
(1, 14, 14, 8)
-4.0184493 2.6935842
(1, 14, 14, 8)
(1, 14, 14, 8)
-4.111106 3.548783
(1, 14, 14, 12)
(1, 14, 14, 12)
-3.6233983 2.8486876
(1, 14, 14, 12)
(1, 14, 14, 12)
-3.9633362 2.785605
(1, 7, 7, 20)
(1, 7, 7, 20)
-2.025126 1.4325218
(1, 7, 7, 20)
(1, 7, 7, 20)
-2.2326288 1.9860396


### Layer-by-layer comparison

In [33]:
buffer = [
    np.random.rand(*[1, 224, 224, 3]),
    np.random.rand(*[1, 56, 56, 3]),
    np.random.rand(*[1, 28, 28, 4]),
    np.random.rand(*[1, 28, 28, 4]),
    np.random.rand(*[1, 14, 14, 8]),
    np.random.rand(*[1, 14, 14, 8]),
    np.random.rand(*[1, 14, 14, 8]),
    np.random.rand(*[1, 14, 14, 12]),
    np.random.rand(*[1, 14, 14, 12]),
    np.random.rand(*[1, 7, 7, 20]),
    np.random.rand(*[1, 7, 7, 20])
]
pth_inputs = [torch.from_numpy(x).permute(0, 3, 1, 2).float() for x in buffer]
keras_inputs = {f'i{i}': buffer[i] for i in range(len(buffer))}

activation = {}
def get_activation(name):
    def hook(model, input, output):
        activation[name] = output.detach()
    return hook

pth_outputs = torch_module(*pth_inputs)

pth_outputs = [
    x.permute(0, 2, 3, 1).detach().numpy() 
    if len(x.shape) > 2 else x.detach().numpy() 
    for x in pth_outputs
]
pth_gesture, pth_buf = pth_outputs[0], pth_outputs[1:]

In [35]:
print('PyTorch: (out, in, h, w)')
print(torch_module.features[0][0].weight.shape)
print(torch_module.features[0][0].weight.permute(2, 3, 1, 0))

PyTorch: (out, in, h, w)
torch.Size([32, 3, 3, 3])
tensor([[[[ 6.2845e-02, -1.6344e-03, -8.0944e-02,  2.9546e-03,  7.4233e-03,
            1.9662e-01, -2.1221e-03, -7.3403e-02, -7.6744e-05, -1.9518e-03,
           -1.3955e-03, -2.6491e-01,  3.1067e-01,  2.3495e-02,  3.1078e-02,
            1.1266e-02, -1.7018e-03, -3.4724e-03,  1.1746e-01,  4.3772e-02,
            5.9811e-03,  2.7355e-01,  1.2261e-03, -1.2525e-01, -5.4357e-03,
            4.5539e-03, -9.2122e-03,  1.2374e-02,  8.1100e-02, -2.3521e-02,
           -9.1651e-03, -1.5124e-01],
          [ 2.1046e-02, -2.1307e-05,  8.9641e-02,  5.6414e-03, -9.9653e-04,
           -1.5683e-01,  1.2183e-02, -6.7451e-02, -4.0242e-03,  2.3432e-03,
            2.0257e-03, -3.4049e-01,  2.2713e-01,  4.4627e-02,  1.7877e-02,
            1.4521e-01, -4.6751e-02, -8.3149e-03,  1.6424e-02,  5.3395e-02,
            7.7508e-04, -1.5610e-01, -1.4899e-04,  2.7810e-01,  1.1604e-03,
           -1.9442e-03, -1.5510e-02, -9.6636e-03,  1.2636e-02,  3.6948e-02,

In [190]:
torch_layer_list = []
for module in torch_module.features:
    try:
        for layer in module:
            torch_layer_list.append(layer)
    except:
        for layer in list(module.modules())[1]:
            torch_layer_list.append(layer)

In [201]:
len(torch_layer_list)

104

In [238]:
torch_convs = []
torch_batchnorm = []
for layer in torch_layer_list:
    if isinstance(layer, torch.nn.ReLU6):
        torch_layer_list.remove(layer)
    if isinstance(layer, torch.nn.Conv2d):
        torch_convs.append(layer)
    if isinstance(layer, torch.nn.BatchNorm2d):
        torch_batchnorm.append(layer)

In [242]:
print('torch conv layers',len(torch_convs))
print('torch batchnorm layers',len(torch_batchnorm))

torch conv layers 52
torch batchnorm layers 52


In [211]:
keras_list = tsm_pth2keras(torch_module, keras_tsm)

features.0.0
features.0.1
features.1.conv.0
features.1.conv.1
features.1.conv.3
features.1.conv.4
features.2.conv.0
features.2.conv.1
features.2.conv.3
features.2.conv.4
features.2.conv.6
features.2.conv.7
features.3.conv.0
features.3.conv.1
features.3.conv.3
features.3.conv.4
features.3.conv.6
features.3.conv.7
features.4.conv.0
features.4.conv.1
features.4.conv.3
features.4.conv.4
features.4.conv.6
features.4.conv.7
features.5.conv.0
features.5.conv.1
features.5.conv.3
features.5.conv.4
features.5.conv.6
features.5.conv.7
features.6.conv.0
features.6.conv.1
features.6.conv.3
features.6.conv.4
features.6.conv.6
features.6.conv.7
features.7.conv.0
features.7.conv.1
features.7.conv.3
features.7.conv.4
features.7.conv.6
features.7.conv.7
features.8.conv.0
features.8.conv.1
features.8.conv.3
features.8.conv.4
features.8.conv.6
features.8.conv.7
features.9.conv.0
features.9.conv.1
features.9.conv.3
features.9.conv.4
features.9.conv.6
features.9.conv.7
features.10.conv.0
features.10.conv.1


In [212]:
len(keras_list)

# drop last dense
del keras_list[-1]

In [213]:
len(keras_list)

104

In [247]:
keras_batchnorm = []
keras_convs = []
for layer in keras_list:
    if isinstance(layer, BatchNormalization):
        keras_batchnorm.append(layer)
    if isinstance(layer, Conv2D):
        keras_convs.append(layer)

In [249]:
print('keras conv layers',len(keras_convs))
print('keras batchnorm layers',len(keras_batchnorm))

keras conv layers 52
keras batchnorm layers 52


### Compare convs

In [359]:
difference_indexes = []
res = 0.0
for i in range(len(keras_convs)):
    prev_res = res
    res +=(torch_convs[i].weight.permute(2, 3, 1, 0).detach().numpy() - keras_convs[i].get_weights()[0]).sum()
    if res != prev_res:
        print('Layer with difference:',i)
        difference_indexes.append(i)

Layer with difference: 1
Layer with difference: 4
Layer with difference: 7
Layer with difference: 13
Layer with difference: 16
Layer with difference: 19
Layer with difference: 22
Layer with difference: 25
Layer with difference: 28
Layer with difference: 31
Layer with difference: 34
Layer with difference: 37
Layer with difference: 40
Layer with difference: 43
Layer with difference: 46
Layer with difference: 49


In [361]:
# check if different layera are depthvise and compute correctly the difference
for i in difference_indexes:
    flattened = [val for sublist in keras_convs[i].get_weights()[0][0][0] for val in sublist]
    dif = (torch_convs[i].weight.permute(2, 3, 1, 0).detach().numpy()[0][0][0] - flattened).sum()
    print(dif)

0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0


### Compare betchnorms

In [290]:
# Identical
res = 0.0
for i in range(len(keras_batchnorm)):
    prev_res = res
    res +=(torch_batchnorm[i].weight.detach().numpy() - keras_batchnorm[i].get_weights()[0]).sum()
    if res != prev_res:
        print('Layer with difference:',i)
print(res)

0.0


### Compare dense

In [375]:
res = (keras_tsm.layers[-11].get_weights()[0] - torch_module.classifier.weight.detach().numpy().T).sum()
print(res)

0.0


* [pitfalls](https://shaoanlu.wordpress.com/2019/05/23/pitfalls-encountered-porting-models-to-keras-from-pytorch-and-tensorflow/)
* [paddings](https://stackoverflow.com/questions/37674306/what-is-the-difference-between-same-and-valid-padding-in-tf-nn-max-pool-of-t/39371113)
* [batchnorm](https://stackoverflow.com/questions/60079783/difference-between-keras-batchnormalization-and-pytorchs-batchnorm2d)
* [output_names](https://github.com/tensorflow/tensorflow/issues/34114)
* [strided slice: layers2graph model](https://github.com/tensorflow/tfjs/issues/2348)